# Day 35：LlamaIndex 核心架构与实战：Document/Node/Index/QueryEngine 与框架选型（Jupyter Notebook 实战手册）

> **所属阶段**：第 3 阶段 · 第三阶段：LangChain 与企业级 RAG 开发 ｜ **第 5 周**：第5周：企业级 RAG 全链路、高级检索优化与评估  
> **GitLab 工单**：`STARMIND-035` ｜ **开发分支**：`feat/day35-llamaindex-engine-benchmark`  
> **配套 HTML5 课件**：`courseware/day35/index.html` ｜ **独立版本快照**：`versions/day35/`

---

## 🎙️ 老陈开场旁白

早啊各位！今天是第 5 周最后一天，咱们把 RAG 生态的另一座大山——**LlamaIndex（原 GPT Index）** 彻底吃透！
很多同学老问我：「老陈，LangChain 和 LlamaIndex 到底啥区别？学完一个还要不要学另一个？」
一句话讲透两者的基因差异：
- **LangChain 的基因是「编排（Orchestration）」**：它把一切抽象成 `Runnable` 管道（LCEL），像乐高一样擅长串起多步推理、工具调用、Agent 工作流；
- **LlamaIndex 的基因是「数据索引（Data Framework）」**：它天生为复杂文档结构而生！它的 `TextNode` 不是孤零零的切片，而是**自带前后节点双向链表（PREVIOUS / NEXT）和父子拓扑关系的图节点**；它的 `Index` 也不只是向量库，还原生提供 `VectorStoreIndex`、`SummaryIndex`、`TreeIndex`、`KnowledgeGraphIndex`，配合 `RouterQueryEngine` 可以自动判断「用户问细节走向量索引、用户问全文总结走摘要索引」！
今天咱们既要把 LlamaIndex 核心引擎亲手写出来跑通，还要拿它和 LangChain 风格管道在星辰语料上做一次硬核基准横评，给明天项目二拍板技术选型！

### 承上启下
- **昨日基线**：昨天（Day 34）我们建好了类 Ragas 四维评估与 CI 门禁体系。但技术委员会在阶段项目二启动前抛来一个经典架构选型题：「做企业级知识库问答，到底是继续用 LangChain，还是换用专门为数据索引而生的 LlamaIndex？别空口白话，拿核心原理和基准数据说话！」
- **明日预告**：明天（Day 36）我们正式进入**阶段项目二（上）——企业级知识库问答系统 StarMind-KB v2.0**，完成架构设计、多格式文档入库、混合检索+Rerank 引擎与 FastAPI 核心 REST API！


In [ ]:
# 环境自检与路径初始化：确保可以无缝导入当日版本模块并支持异步协程运行
import os
import sys
from pathlib import Path

# 允许在 Jupyter 已有事件循环内安全调用 asyncio.run()
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'versions').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DAY_DIR = PROJECT_ROOT / 'versions' / 'day35'
for _p in (DAY_DIR, DAY_DIR / 'src', PROJECT_ROOT):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
os.chdir(DAY_DIR)
__file__ = str(DAY_DIR / 'src' / 'notebook_entry.py')

print(f'✅ Python 版本: {sys.version.split()[0]}')
print(f'✅ 项目根目录: {PROJECT_ROOT}')
print(f'✅ 当日版本目录: {DAY_DIR}')


## 📋 企业需求背景与验收目标（STARMIND-035）

项目二「企业级知识库问答系统」启动前，架构评审组要求对 LangChain 路线与 LlamaIndex 路线进行原理剖析与小规模 PoC 基准测试，明确两者的适用场景与最终选型结论。

**今日迭代目标：**
- 完整实现 LlamaIndex 核心对象模型：`Document`、`TextNode`（含 `SOURCE / PREVIOUS / NEXT` 节点关系）、`SentenceNodeParser`。
- 实现双索引结构：`VectorStoreIndex`（细节精准检索 + 前后节点窗口自动扩展）与 `SummaryIndex`（全节点扫描摘要），并实现 `RouterQueryEngine` 自动路由。
- 编写基准测试脚本，对比 LlamaIndex 风格（Node 拓扑扩展）与 LangChain 风格（扁平切片管道）在上下文完整度、召回率、延迟与开发复杂度上的表现。

**验收标准：**
- `SentenceNodeParser` 生成的 `TextNode` 链表必须正确维护 `PREVIOUS` 与 `NEXT` 指针及 `SOURCE` 原文档 ID。
- `RouterQueryEngine` 能自动将「总结/概览」类问题路由至 `SummaryIndex`，将「具体金额/流程」类问题路由至 `VectorStoreIndex`。
- `ResponseSynthesizer` 支持 `compact` 与 `refine` 两种响应合成模式。
- 所有 pytest 自动化测试 100% 本地离线通过。

---

## 📚 课堂核心技术笔记

### 1. 1. LangChain vs LlamaIndex 架构选型红宝书

- **首选 LlamaIndex 的场景**：核心诉求是「高难度、多模态、超长/复杂结构文档检索与问答」（如财报表格、法律合同、百页技术白皮书），需要开箱即用的 `SentenceWindowNodeParser`、`AutoMergingRetriever`、`SubQuestionQueryEngine`、`RouterQueryEngine`。
- **首选 LangChain / LangGraph 的场景**：核心诉求是「复杂多步骤工作流编排、多工具调用、多 Agent 状态机协同、人工介入审批（HITL）」以及需要极高定制自由度的全栈 AI 平台。
- **大厂真实做法**：不要二极管二选一！很多团队用 LlamaIndex 的文档解析与多级索引作为检索 Tool，挂到 LangGraph Agent 里统一调度。

> **⚠️ 老陈避坑**：踩坑提醒：LlamaIndex 在 `v0.10+` 进行了彻底的模块化拆分，旧教程里的 `from llama_index import VectorStoreIndex` 已废弃，生产环境务必使用 `from llama_index.core import VectorStoreIndex, Document` 及独立插件包 `llama-index-llms-openai`、`llama-index-embeddings-huggingface`。

### 2. 2. 生产环境官方 LlamaIndex 5 行极简代码对照

LlamaIndex 最大的工程爽点就是高内聚封装，5 行代码即可跑起一套完整知识库：
```python
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader

documents = SimpleDirectoryReader('./data/starmind_docs').load_data()
index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine(similarity_top_k=3, response_mode='compact')
response = query_engine.query('实习生去上海出差住宿上限是多少？')
print(response.response, response.source_nodes)
```

> **⚠️ 老陈避坑**：踩坑提醒：高封装是一把双刃剑——上手极快，但当线上出现奇葩 Bad Case 需要魔改底层 Prompt 或检索过滤逻辑时，一定要善用 `CallbackManager` / `LlamaDebugHandler` 打印出内部实际拼出来的 Prompt 和召回 Node，切忌当黑盒用！


## 💻 实操步骤 1：步骤一：手写 LlamaIndex 核心引擎（Document / TextNode 链表 / 双 Index / RouterQueryEngine）

- **目标文件**：`versions/day35/src/day35_llamaindex_core.py`
- **老陈代码旁白**：为了让大家真正看透 LlamaIndex 内部到底是怎么运转的，咱们用不到 240 行纯 Python 还原它的核心骨架：带 `PREVIOUS/NEXT` 双向指针的 `TextNode`、`SentenceNodeParser`、`VectorStoreIndex`、`SummaryIndex`、`ResponseSynthesizer` 和 `RouterQueryEngine`！


In [ ]:
"""
Day 35 - 步骤一：LlamaIndex 核心架构精简实现
还原：Document -> SentenceNodeParser -> TextNode(关系链表) -> VectorStoreIndex / SummaryIndex
      -> ResponseSynthesizer(compact/refine) -> RouterQueryEngine
"""

from __future__ import annotations

import hashlib
import math
import re
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Sequence, Tuple


@dataclass
class Document:
    """LlamaIndex 风格原始文档对象。"""

    text: str
    doc_id: str = ""
    metadata: Dict[str, Any] = field(default_factory=dict)

    def __post_init__(self) -> None:
        if not self.doc_id:
            self.doc_id = "doc_" + hashlib.md5(self.text.encode("utf-8")).hexdigest()[:8]


@dataclass
class TextNode:
    """
    LlamaIndex 核心知识节点：
    不仅保存文本与元数据，还通过 relationships 维护 SOURCE / PREVIOUS / NEXT 拓扑指针。
    """

    text: str
    node_id: str
    metadata: Dict[str, Any] = field(default_factory=dict)
    relationships: Dict[str, str] = field(default_factory=dict)


@dataclass
class NodeWithScore:
    node: TextNode
    score: float


@dataclass
class QueryResponse:
    response: str
    source_nodes: List[NodeWithScore]
    routed_engine: str = "vector"


class SentenceNodeParser:
    """按句切分并自动串联 PREVIOUS / NEXT 双向链表的节点解析器。"""

    def __init__(self, sentences_per_node: int = 1) -> None:
        self.sentences_per_node = max(1, sentences_per_node)

    def get_nodes_from_documents(self, documents: Sequence[Document]) -> List[TextNode]:
        all_nodes: List[TextNode] = []
        for doc in documents:
            sents = [s.strip() for s in re.split(r"(?<=[。！？\n])", doc.text) if s.strip()]
            doc_nodes: List[TextNode] = []
            for idx in range(0, len(sents), self.sentences_per_node):
                chunk_text = "".join(sents[idx : idx + self.sentences_per_node])
                node_id = f"{doc.doc_id}_n{len(doc_nodes)}"
                node = TextNode(
                    text=chunk_text,
                    node_id=node_id,
                    metadata={**doc.metadata, "doc_id": doc.doc_id, "node_idx": len(doc_nodes)},
                    relationships={"SOURCE": doc.doc_id},
                )
                doc_nodes.append(node)

            # 串联同一文档内相邻节点的 PREVIOUS / NEXT 双向指针
            for i, node in enumerate(doc_nodes):
                if i > 0:
                    node.relationships["PREVIOUS"] = doc_nodes[i - 1].node_id
                if i < len(doc_nodes) - 1:
                    node.relationships["NEXT"] = doc_nodes[i + 1].node_id

            all_nodes.extend(doc_nodes)
        return all_nodes


def _embed_text(text: str, dim: int = 96) -> List[float]:
    vec = [0.0] * dim
    lowered = text.lower()
    tokens = re.findall(r"[a-z0-9]+(?:[-_][a-z0-9]+)*", lowered)
    for chunk in re.findall(r"[\u4e00-\u9fff]+", lowered):
        for ch in chunk:
            tokens.append(ch)
        for i in range(len(chunk) - 1):
            tokens.append(chunk[i : i + 2])
    for tok in tokens:
        h = int(hashlib.md5(tok.encode("utf-8")).hexdigest(), 16)
        vec[h % dim] += 1.0
    norm = math.sqrt(sum(v * v for v in vec)) or 1.0
    return [v / norm for v in vec]


def _cosine(a: Sequence[float], b: Sequence[float]) -> float:
    return sum(x * y for x, y in zip(a, b))


class ResponseSynthesizer:
    """LlamaIndex 风格响应合成器，支持 compact（紧凑拼装）与 refine（逐块精炼）模式。"""

    def __init__(self, response_mode: str = "compact") -> None:
        if response_mode not in ("compact", "refine"):
            raise ValueError(f"Unsupported response_mode: {response_mode}")
        self.response_mode = response_mode

    def synthesize(self, query: str, nodes: Sequence[NodeWithScore]) -> str:
        if not nodes:
            return "知识库中未检索到与当前问题直接相关的制度条款。"

        if self.response_mode == "compact":
            evidences = [
                f"[{i}] ({n.node.metadata.get('source', '未知')}) {n.node.text}"
                for i, n in enumerate(nodes, start=1)
            ]
            return f"【Compact综合回答】针对「{query}」，依据以下制度条款：\n" + "\n".join(evidences)

        # refine 模式：逐个节点增量精炼摘要（典型用于 SummaryIndex 全局总结）
        refined_points: List[str] = []
        for idx, n in enumerate(nodes, start=1):
            source = n.node.metadata.get("source", "文档")
            refined_points.append(f"要点{idx}（{source}）：{n.node.text}")
        return f"【Refine全量摘要】针对「{query}」，汇总全文核心要点如下：\n" + "\n".join(refined_points)


class VectorStoreIndex:
    """LlamaIndex 风格向量索引：支持邻域节点扩展（expand_prev_next=True）。"""

    def __init__(self, nodes: Sequence[TextNode]) -> None:
        self.nodes: List[TextNode] = list(nodes)
        self.node_map: Dict[str, TextNode] = {n.node_id: n for n in self.nodes}
        self.vectors: Dict[str, List[float]] = {n.node_id: _embed_text(n.text) for n in self.nodes}

    @classmethod
    def from_documents(
        cls,
        documents: Sequence[Document],
        parser: Optional[SentenceNodeParser] = None,
    ) -> "VectorStoreIndex":
        node_parser = parser or SentenceNodeParser(sentences_per_node=1)
        return cls(node_parser.get_nodes_from_documents(documents))

    def retrieve(
        self,
        query: str,
        similarity_top_k: int = 2,
        expand_prev_next: bool = True,
    ) -> List[NodeWithScore]:
        q_vec = _embed_text(query)
        scored = [
            NodeWithScore(node=n, score=round(_cosine(q_vec, self.vectors[n.node_id]), 4))
            for n in self.nodes
        ]
        scored.sort(key=lambda x: x.score, reverse=True)
        top_hits = scored[:similarity_top_k]

        if not expand_prev_next:
            return top_hits

        # 利用 TextNode.relationships 的 PREVIOUS 指针自动补全前文上下文，消除「上述标准」指代断层
        expanded: List[NodeWithScore] = []
        for hit in top_hits:
            prev_id = hit.node.relationships.get("PREVIOUS")
            merged_text = hit.node.text
            if prev_id and prev_id in self.node_map:
                merged_text = self.node_map[prev_id].text + merged_text
            expanded.append(
                NodeWithScore(
                    node=TextNode(
                        text=merged_text,
                        node_id=hit.node.node_id,
                        metadata={**hit.node.metadata, "expanded_prev": bool(prev_id)},
                        relationships=dict(hit.node.relationships),
                    ),
                    score=hit.score,
                )
            )
        return expanded

    def as_query_engine(
        self,
        similarity_top_k: int = 2,
        response_mode: str = "compact",
        expand_prev_next: bool = True,
    ) -> "RetrieverQueryEngine":
        return RetrieverQueryEngine(
            retriever_fn=lambda q: self.retrieve(
                q, similarity_top_k=similarity_top_k, expand_prev_next=expand_prev_next
            ),
            synthesizer=ResponseSynthesizer(response_mode=response_mode),
            engine_name="vector_index",
        )


class SummaryIndex:
    """LlamaIndex 风格摘要索引：遍历全量节点用于全局归纳总结。"""

    def __init__(self, nodes: Sequence[TextNode]) -> None:
        self.nodes = list(nodes)

    @classmethod
    def from_documents(
        cls,
        documents: Sequence[Document],
        parser: Optional[SentenceNodeParser] = None,
    ) -> "SummaryIndex":
        node_parser = parser or SentenceNodeParser(sentences_per_node=1)
        return cls(node_parser.get_nodes_from_documents(documents))

    def as_query_engine(self, response_mode: str = "refine") -> "RetrieverQueryEngine":
        return RetrieverQueryEngine(
            retriever_fn=lambda _q: [NodeWithScore(node=n, score=1.0) for n in self.nodes],
            synthesizer=ResponseSynthesizer(response_mode=response_mode),
            engine_name="summary_index",
        )


class RetrieverQueryEngine:
    def __init__(self, retriever_fn, synthesizer: ResponseSynthesizer, engine_name: str) -> None:
        self.retriever_fn = retriever_fn
        self.synthesizer = synthesizer
        self.engine_name = engine_name

    def query(self, question: str) -> QueryResponse:
        nodes = self.retriever_fn(question)
        answer = self.synthesizer.synthesize(question, nodes)
        return QueryResponse(response=answer, source_nodes=nodes, routed_engine=self.engine_name)


class RouterQueryEngine:
    """意图路由查询引擎：自动判断走 SummaryIndex 全局总结还是 VectorStoreIndex 细节精查。"""

    SUMMARY_TRIGGERS = ("总结", "概览", "全文", "要点", "归纳", "都有哪些规定", "梳理一下")

    def __init__(
        self,
        vector_engine: RetrieverQueryEngine,
        summary_engine: RetrieverQueryEngine,
    ) -> None:
        self.vector_engine = vector_engine
        self.summary_engine = summary_engine

    def query(self, question: str) -> QueryResponse:
        if any(kw in question for kw in self.SUMMARY_TRIGGERS):
            return self.summary_engine.query(question)
        return self.vector_engine.query(question)


if __name__ == "__main__":
    doc = Document(
        text=(
            "一线城市（北京、上海、深圳、广州）出差酒店住宿报销上限为每晚600元。"
            "上述标准同样适用于外包人员及在岗实习生。"
            "出差餐饮补助按每天100元定额随月薪发放，无需粘贴发票。"
        ),
        doc_id="fin_travel_2026",
        metadata={"source": "财务差旅制度_2026.md"},
    )
    vec_idx = VectorStoreIndex.from_documents([doc])
    sum_idx = SummaryIndex.from_documents([doc])
    router = RouterQueryEngine(
        vector_engine=vec_idx.as_query_engine(similarity_top_k=1, expand_prev_next=True),
        summary_engine=sum_idx.as_query_engine(response_mode="refine"),
    )

    r_detail = router.query("实习生出差住宿适用什么标准？")
    print(f"[路由={r_detail.routed_engine}]\n{r_detail.response}\n")

    r_summary = router.query("请总结一下这份差旅制度的全部要点")
    print(f"[路由={r_summary.routed_engine}]\n{r_summary.response}")


### 🔍 步骤 1 代码深度解析

1. 看细节提问「实习生出差住宿适用什么标准？」：虽然只有第 2 句提到了「实习生」，原文里写的是「上述标准同样适用于外包人员及在岗实习生」，没有写金额；因为 `TextNode` 自动维护了 `PREVIOUS` 指针，`VectorStoreIndex` 一键把第 1 句的「每晚600元」自动拼回来了！
2. 看总结提问「请总结一下这份差旅制度的全部要点」：`RouterQueryEngine` 自动路由到 `SummaryIndex` 并调用 `refine` 模式遍历全节点，避免传统 Top-K 漏掉后半篇内容。

**预期运行输出参考：**
```text
[路由=vector_index]
【Compact综合回答】针对「实习生出差住宿适用什么标准？」，依据以下制度条款：
[1] (财务差旅制度_2026.md) 一线城市（北京、上海、深圳、广州）出差酒店住宿报销上限为每晚600元。上述标准同样适用于外包人员及在岗实习生。

[路由=summary_index]
【Refine全量摘要】针对「请总结一下这份差旅制度的全部要点」，汇总全文核心要点如下：
要点1（财务差旅制度_2026.md）：一线城市...
要点2（财务差旅制度_2026.md）：上述标准同样适用于...
要点3（财务差旅制度_2026.md）：出差餐饮补助按每天100元定额随月薪发放，无需粘贴发票。
```


## 💻 实操步骤 2：步骤二：LlamaIndex vs LangChain 风格基准横评与架构选型器

- **目标文件**：`versions/day35/src/day35_framework_benchmark.py`
- **老陈代码旁白**：空谈选型没有说服力！咱们拿同一个星辰企业文档集，在「跨段指代查询」和「全文摘要查询」上实测对比 LangChain 扁平切片管道与 LlamaIndex 节点图路由引擎的答案完整度与响应耗时。


In [ ]:
"""
Day 35 - 步骤二：LlamaIndex 风格 vs LangChain 风格扁平管道基准对比评测
"""

from __future__ import annotations

import sys
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List

ROOT = Path(__file__).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.day35_llamaindex_core import (
    Document,
    RouterQueryEngine,
    SummaryIndex,
    VectorStoreIndex,
)


@dataclass
class BenchmarkRow:
    framework: str
    completeness_rate: float
    avg_latency_ms: float
    strengths: str


BENCHMARK_DOCS = [
    Document(
        text=(
            "一线城市（北京、上海、深圳、广州）出差酒店住宿报销上限为每晚600元。"
            "上述标准同样适用于外包人员及在岗实习生。"
            "出差餐饮补助按每天100元定额随月薪发放，无需粘贴发票。"
        ),
        doc_id="fin_travel",
        metadata={"source": "差旅制度.md"},
    ),
    Document(
        text=(
            "生产环境日常发布窗口为周一至周四10:00-18:00，严禁周五17:00后非紧急变更。"
            "若遇P0级线上故障需打破该窗口限制，必须经值班架构师与CTO双人审批。"
        ),
        doc_id="dev_release",
        metadata={"source": "研发发布规范.md"},
    ),
]

# 专门测试「跨段指代」与「全文总结」两类高难场景
BENCHMARK_CASES = [
    ("实习生出差住宿适用什么标准？", ["600元", "实习生"]),
    ("P0级线上故障打破发布窗口限制需要谁审批，原窗口是哪几天？", ["周一至周四", "CTO"]),
    ("请总结一下全部制度的核心要点", ["600元", "100元", "周一至周四"]),
]


def run_framework_benchmark() -> List[BenchmarkRow]:
    # 1. 模拟 LangChain 经典扁平切片检索（无 PREVIOUS 指针扩展、无 SummaryIndex 路由）
    flat_index = VectorStoreIndex.from_documents(BENCHMARK_DOCS)
    lc_flat_engine = flat_index.as_query_engine(
        similarity_top_k=1, response_mode="compact", expand_prev_next=False
    )

    # 2. LlamaIndex 风格：TextNode 邻域自动扩展 + RouterQueryEngine(Vector + Summary)
    li_router = RouterQueryEngine(
        vector_engine=flat_index.as_query_engine(
            similarity_top_k=1, response_mode="compact", expand_prev_next=True
        ),
        summary_engine=SummaryIndex.from_documents(BENCHMARK_DOCS).as_query_engine(
            response_mode="refine"
        ),
    )

    rows: List[BenchmarkRow] = []
    for name, engine, strength in [
        ("LangChain_Flat_RAG", lc_flat_engine, "LCEL灵活编排、生态最全、多Agent工作流强"),
        ("LlamaIndex_NodeRouter", li_router, "Node链表防断层、多态Index与全局摘要天然占优"),
    ]:
        passed = 0
        t0 = time.perf_counter()
        for q, must_keywords in BENCHMARK_CASES:
            resp = engine.query(q)
            if all(kw in resp.response for kw in must_keywords):
                passed += 1
        elapsed_ms = (time.perf_counter() - t0) * 1000.0 / len(BENCHMARK_CASES)
        rows.append(
            BenchmarkRow(
                framework=name,
                completeness_rate=round(passed / len(BENCHMARK_CASES), 4),
                avg_latency_ms=round(elapsed_ms, 3),
                strengths=strength,
            )
        )
    return rows


def recommend_framework(need_multi_agent: bool, heavy_doc_summary: bool) -> Dict[str, str]:
    """企业级技术选型决策辅助器。"""
    if heavy_doc_summary and not need_multi_agent:
        return {"choice": "LlamaIndex", "reason": "重度长文档索引、多级摘要与复杂数据连接场景最优"}
    if need_multi_agent and not heavy_doc_summary:
        return {"choice": "LangChain + LangGraph", "reason": "重度工具编排、状态机工作流与多 Agent 协作最优"}
    return {
        "choice": "Hybrid (LlamaIndex 索引检索 + LangGraph 上层编排)",
        "reason": "兼具强文档索引能力与复杂工作流编排，是中大型企业中台常见组合",
    }


if __name__ == "__main__":
    print("=== LangChain 扁平管道 vs LlamaIndex 节点路由基准对比 ===")
    for row in run_framework_benchmark():
        print(
            f"  {row.framework:22s} | 答案完整率={row.completeness_rate:6.2%} "
            f"| 平均耗时={row.avg_latency_ms:6.3f}ms | 优势: {row.strengths}"
        )
    print("\n星辰智脑项目二选型建议:", recommend_framework(need_multi_agent=True, heavy_doc_summary=True))


### 🔍 步骤 2 代码深度解析

1. 在包含「跨段指代（实习生适用上述标准）」「条件回溯（打破该窗口限制）」和「全文要点总结」的高难评测集上，LlamaIndex 的 Node 链表扩展 + RouterQueryEngine 答案完整率达到 100%，直观拉开了与普通扁平切片管道的差距。
2. `recommend_framework` 明确了工程界共识：不盲目站队，按业务复杂度选择单框架或「LlamaIndex 检索 + LangGraph 编排」组合架构。

**预期运行输出参考：**
```text
=== LangChain 扁平管道 vs LlamaIndex 节点路由基准对比 ===
  LangChain_Flat_RAG     | 答案完整率= 0.00% | 平均耗时= 0.215ms | 优势: LCEL灵活编排、生态最全、多Agent工作流强
  LlamaIndex_NodeRouter  | 答案完整率=100.00% | 平均耗时= 0.264ms | 优势: Node链表防断层、多态Index与全局摘要天然占优

星辰智脑项目二选型建议: {'choice': 'Hybrid (LlamaIndex 索引检索 + LangGraph 上层编排)', 'reason': '兼具强文档索引能力与复杂工作流编排，是中大型企业中台常见组合'}
```


## 🧪 当日自动化单元测试

在终端运行 `pytest versions/day35/tests/test_day35.py -v` 即可验收全部测试断言。下面展示完整测试脚本源码，供对照学习测试写法：

```python
"""Day 35 自动化单元测试：验证 LlamaIndex Node 链表、双索引路由与框架基准对比。"""

from src.day35_framework_benchmark import recommend_framework, run_framework_benchmark
from src.day35_llamaindex_core import (
    Document,
    RouterQueryEngine,
    SentenceNodeParser,
    SummaryIndex,
    VectorStoreIndex,
)


def test_sentence_node_parser_builds_doubly_linked_relationships():
    doc = Document("第一句规定标准。第二句补充例外。第三句说明审批。", doc_id="d_test")
    nodes = SentenceNodeParser(sentences_per_node=1).get_nodes_from_documents([doc])
    assert len(nodes) == 3
    assert nodes[0].relationships["SOURCE"] == "d_test"
    assert "PREVIOUS" not in nodes[0].relationships
    assert nodes[0].relationships["NEXT"] == nodes[1].node_id
    assert nodes[1].relationships["PREVIOUS"] == nodes[0].node_id
    assert nodes[1].relationships["NEXT"] == nodes[2].node_id
    assert nodes[2].relationships["PREVIOUS"] == nodes[1].node_id
    assert "NEXT" not in nodes[2].relationships


def test_router_query_engine_routes_vector_and_summary():
    doc = Document(
        "一线城市出差住宿上限每晚600元。上述标准同样适用于实习生。餐饮补助每天100元。",
        doc_id="fin01",
        metadata={"source": "差旅制度.md"},
    )
    vec_idx = VectorStoreIndex.from_documents([doc])
    sum_idx = SummaryIndex.from_documents([doc])
    router = RouterQueryEngine(
        vector_engine=vec_idx.as_query_engine(similarity_top_k=1, expand_prev_next=True),
        summary_engine=sum_idx.as_query_engine(response_mode="refine"),
    )

    detail_resp = router.query("实习生出差住宿适用什么标准？")
    assert detail_resp.routed_engine == "vector_index"
    assert "600元" in detail_resp.response and "实习生" in detail_resp.response

    summary_resp = router.query("请总结这份差旅制度的全部要点")
    assert summary_resp.routed_engine == "summary_index"
    assert "600元" in summary_resp.response and "100元" in summary_resp.response


def test_framework_benchmark_shows_node_router_advantage():
    rows = run_framework_benchmark()
    by_name = {r.framework: r for r in rows}
    assert by_name["LlamaIndex_NodeRouter"].completeness_rate > by_name["LangChain_Flat_RAG"].completeness_rate
    assert by_name["LlamaIndex_NodeRouter"].completeness_rate == 1.0

    rec = recommend_framework(need_multi_agent=True, heavy_doc_summary=True)
    assert "Hybrid" in rec["choice"]

```


## 📝 课后梯度实战作业与标准答案

建议先在下方新建空白单元格独立完成编码，运行通过后再核对参考答案。


### 作业 1（基础巩固）：作业一：实现双向邻域扩展器（同时拉取 PREVIOUS + CURRENT + NEXT）

**题目要求：**
编写函数 `expand_bidirectional(node_map, hit_node)`：给定 `node_map` 和命中的 `TextNode`，若存在 `PREVIOUS` 和 `NEXT` 节点，按「前文 + 当前 + 后文」顺序拼接并返回完整上下文。

**思路提示：** 分别从 `hit_node.relationships.get('PREVIOUS')` 与 `get('NEXT')` 查找节点，过滤掉 `None` 后拼接 `text`。


In [ ]:
"""Day 35 作业一参考答案：TextNode 双向邻域扩展器"""
from typing import Dict
from src.day35_llamaindex_core import Document, SentenceNodeParser, TextNode


def expand_bidirectional(node_map: Dict[str, TextNode], hit_node: TextNode) -> str:
    parts = []
    prev_id = hit_node.relationships.get("PREVIOUS")
    next_id = hit_node.relationships.get("NEXT")
    if prev_id and prev_id in node_map:
        parts.append(node_map[prev_id].text)
    parts.append(hit_node.text)
    if next_id and next_id in node_map:
        parts.append(node_map[next_id].text)
    return "".join(parts)


if __name__ == "__main__":
    doc = Document("一线城市住宿上限600元。上述标准适用于实习生。超标需经VP审批。", doc_id="d1")
    nodes = SentenceNodeParser(1).get_nodes_from_documents([doc])
    nmap = {n.node_id: n for n in nodes}
    merged = expand_bidirectional(nmap, nodes[1])
    print("双向扩展上下文:", merged)
    assert "600元" in merged and "VP审批" in merged


**答案解析与评分标准：**

双向邻域扩展在条款前后都有条件限定（前面定义标准、后面写例外审批）时特别管用。


### 作业 2（企业实战）：作业二：实现 LlamaIndex 风格 MetadataPostprocessor（元数据过滤后处理器）

**题目要求：**
实现 `DepartmentNodePostprocessor(allowed_dept)`：接收 `List[NodeWithScore]`，仅保留 `metadata['department']` 等于 `allowed_dept` 或 `'public'` 的节点，模拟企业知识库部门权限隔离。

**思路提示：** 遍历 `nodes`，检查 `n.node.metadata.get('department', 'public') in (allowed_dept, 'public')`。


In [ ]:
"""Day 35 作业二参考答案：基于部门权限的 NodePostprocessor"""
from typing import List, Sequence
from src.day35_llamaindex_core import NodeWithScore, TextNode


class DepartmentNodePostprocessor:
    def __init__(self, allowed_dept: str) -> None:
        self.allowed_dept = allowed_dept

    def postprocess_nodes(self, nodes: Sequence[NodeWithScore]) -> List[NodeWithScore]:
        return [
            n
            for n in nodes
            if n.node.metadata.get("department", "public") in (self.allowed_dept, "public")
        ]


if __name__ == "__main__":
    sample_nodes = [
        NodeWithScore(TextNode("公共差旅制度", "n1", {"department": "public"}), 0.9),
        NodeWithScore(TextNode("研发架构机密", "n2", {"department": "rd"}), 0.85),
        NodeWithScore(TextNode("财务薪酬明细", "n3", {"department": "finance"}), 0.88),
    ]
    filtered = DepartmentNodePostprocessor("rd").postprocess_nodes(sample_nodes)
    print("研发员工可见节点:", [n.node.node_id for n in filtered])
    assert [n.node.node_id for n in filtered] == ["n1", "n2"]


**答案解析与评分标准：**

NodePostprocessor 是 LlamaIndex 极具扩展性的钩子层，重排、去重、权限过滤、时间衰减加权都可以作为 Postprocessor 链式插拔。


### 作业 3（架构进阶）：作业三：实现 SubQuestionQueryEngine（跨文档子问题拆解引擎）

**题目要求：**
面向多个按主题拆分的 `VectorStoreIndex`（如财务索引 + 研发索引），实现 `SubQuestionQueryEngine`：当用户问跨部门复合问题（「北京住宿上限多少，以及周五晚上能不能发版？」）时，自动拆分子问题分别查询并合并引用。

**思路提示：** 按连接词（`以及/另外/，/和`）拆分子问题，分别调用 `vector_engine.query(sub_q)` 汇总回答与 `source_nodes`。


In [ ]:
"""Day 35 作业三参考答案：SubQuestionQueryEngine 跨主题子问题拆解"""
import re
from typing import List
from src.day35_llamaindex_core import Document, QueryResponse, VectorStoreIndex


class SubQuestionQueryEngine:
    def __init__(self, vector_index: VectorStoreIndex) -> None:
        self.engine = vector_index.as_query_engine(similarity_top_k=1)

    def query(self, complex_q: str) -> QueryResponse:
        sub_qs = [s.strip() for s in re.split(r"以及|另外|[，,；;]", complex_q) if len(s.strip()) >= 4]
        answers: List[str] = []
        all_nodes = []
        for idx, sq in enumerate(sub_qs, start=1):
            r = self.engine.query(sq)
            answers.append(f"子问题{idx}({sq}) -> {r.source_nodes[0].node.text}")
            all_nodes.extend(r.source_nodes)
        return QueryResponse(response="\n".join(answers), source_nodes=all_nodes, routed_engine="sub_question")


if __name__ == "__main__":
    docs = [
        Document("北京出差酒店住宿上限每晚600元。", doc_id="fin", metadata={"source": "财务制度.md"}),
        Document("生产环境严禁周五17:00之后非紧急发版。", doc_id="dev", metadata={"source": "研发规范.md"}),
    ]
    sq_engine = SubQuestionQueryEngine(VectorStoreIndex.from_documents(docs))
    res = sq_engine.query("北京出差住宿上限多少，以及周五晚上能不能发版？")
    print(res.response)
    assert "600元" in res.response and "周五" in res.response


**答案解析与评分标准：**

SubQuestionQueryEngine 是 LlamaIndex 解决多跳、跨文档对比分析（如对比 A 季度与 B 季度财报）的招牌利器。


---

## 🎯 老陈收尾复盘

今天这场 LangChain vs LlamaIndex 深度实战对比，相信大家心里彻底有底了！在真实国内大厂项目里，两者从来不是非此即彼：文档解析、多级索引与复杂检索可以借鉴 LlamaIndex 的 Node 拓扑与多索引路由思想，而上层 API 编排与多 Agent 协同则由 LangChain/LangGraph 统筹。明天（Day 36）正式开工阶段项目二！
